# 02g — `S_unpaired_scaled`: the arm that actually tests P3

**Post-hoc. Decided 2026-08-24, after the `prereg-v2` tag, and it must be labelled so
wherever it is reported.** Estimated **~2.2 h** on a T4. Suggested kernel: `emocap-unpaired-scaled`.

## Why this arm exists

The registered P3 compares `S_paired5` against `S_unpaired` and moves three things at once
— 9× the cells, twice the images, and the paired structure. The prereg says as much: it
calls P3a "a sanity check on the extra data".

`S_paired_matched` (notebook 02f) attacked one half of that. It held cells at 4,390 and kept
the pairing, and it **went to the floor** — accuracy 0.2018, margin −0.0129. Its
distinctiveness explains why: self-BLEU **0.86** against 0.29–0.37 for the working arms, so
the model wrote one caption per image and reused it for all five registers. Conclusion:
volume is *necessary*.

That leaves the other half untested. **Does the volume work without the pairing?** Until
this arm exists, "more data helps" and "parallel data helps" are the same observation, and
the second claim — the one that would generalise past image captioning — is unsupported.

## The contrast

| | images | cells | the 5 cells of one image are… |
|---|---|---|---|
| `S_paired5` | 8,047 | 40,235 | ONE source caption in **FIVE registers** |
| **`S_unpaired_scaled`** | 8,047 | 40,235 | **FIVE source captions** in ONE register |

Same images, same cell count, same cells per image, same corpus, same training budget. The
only difference is whether an image's five cells vary by *register* or by *source text*.
This is the single-variable contrast the registered P3 could not be.

It also extends `S_unpaired` into a volume ladder at constant unpaired structure — 4,390 →
40,235 cells, register contrast absent at both ends — because the 4,390 images `S_unpaired`
used keep the register it assigned them. `scripts/build_posthoc_arms.py` asserts all six
containments and refuses to write the arm otherwise.

## How to read the result

* **Floor, like `S_paired_matched`** → pairing is isolated. Volume alone is not enough, and
  the parallel-data claim is real.
* **Reaches `S_paired5`'s +0.086 margin** → volume alone is sufficient, pairing contributes
  nothing, and the honest headline is "more data helps". Report that.
* **Somewhere between** → both matter, and the split is quantified.

All three outcomes are publishable, and the third is the most likely. Deciding *now* how
each will be read is the point of writing them down before the run.

## Setup

**Attach:** dataset `emocap-v2-arms` — must be the **2026-08-24 or later** push, which is
the first to carry `S_unpaired_scaled.jsonl`. **Accelerator:** GPU **T4 x2**. **Internet:** on.

> P100 will not work — sm_60 against a PyTorch build shipping sm_70 and up.

Resumable: a re-run skips whatever already landed.

Pull the results:

    kaggle kernels output <owner>/emocap-unpaired-scaled -p tmp/emocap-unpaired-scaled \
        --page-size 200 --file-pattern 'S_unpaired_scaled'


In [ ]:
# ── parameters ────────────────────────────────────────────────────────────
RUNS = [
    ("S_unpaired_scaled", 0, False),
    ("S_unpaired_scaled", 1, False),
    ("S_unpaired_scaled", 2, False),
    ("S_unpaired_scaled", 3, False),
    ("S_unpaired_scaled", 4, False),
    ("S_unpaired_scaled", 0, True),   # the registered manipulation check
]
SEED = 42

DATA = "/kaggle/input/emocap-v2-arms"
OUT = "/kaggle/working/runs"


In [ ]:
# Clone the EXACT commit the data was built from. Pinning to the commit recorded in
# provenance.json is what stops a notebook from pairing this dataset version with a
# different version of the code -- a mismatch that would be silent and unrecoverable.
import json, os, subprocess
from pathlib import Path

COMMIT = json.loads(Path(DATA, "provenance.json").read_text())["git_commit"]
if not Path("/kaggle/working/EmoCap").exists():
    subprocess.run(["git", "clone", "-q",
                    "https://github.com/asjad2401/EmoCap.git",
                    "/kaggle/working/EmoCap"], check=True)
subprocess.run(["git", "-C", "/kaggle/working/EmoCap", "checkout", "-q", COMMIT], check=True)
print("code at", COMMIT[:12])


In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/EmoCap/src")
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# Fail here rather than after the first fold. This arm did not exist before 2026-08-24, so
# an older dataset version carries every other arm and not this one.
arm_file = Path(DATA, "arms", "S_unpaired_scaled.jsonl")
if not arm_file.exists():
    raise SystemExit(
        "S_unpaired_scaled.jsonl is not in the attached dataset.\n"
        "Run `uv run python scripts/build_posthoc_arms.py` locally, then\n"
        "`uv run python scripts/push_kaggle.py --part data`, then attach the new version.")
n = sum(1 for _ in arm_file.open())
print(f"{arm_file.name}: {n:,} cells")

# The arm is worthless if the register contrast leaked back in, so it is checked HERE too,
# against the file actually attached -- not only at build time on another machine.
import collections
regs = collections.defaultdict(set)
idxs = collections.defaultdict(set)
for line in arm_file.open():
    r = json.loads(line)
    regs[r["image_id"]].add(r["emotion"])
    idxs[r["image_id"]].add(r["caption_idx"])
bad_r = [i for i, v in regs.items() if len(v) != 1]
bad_k = [i for i, v in idxs.items() if len(v) != 5]
if bad_r or bad_k:
    raise SystemExit(f"arm is malformed: {len(bad_r)} images with != 1 register, "
                     f"{len(bad_k)} with != 5 source captions")
print(f"verified: {len(regs):,} images, exactly 1 register and 5 source captions each")

man = Path(DATA, "arms", "posthoc_manifest.json")
if man.exists():
    print(json.dumps(json.loads(man.read_text())["arms"]["S_unpaired_scaled"]["nested_in"],
                     indent=2))


In [ ]:
import time

results = []
t_all = time.time()

for i, (arm, fold, nc) in enumerate(RUNS, 1):
    tag = f"{arm}-f{fold}" + ("-nc" if nc else "")
    run_dir = Path(OUT, tag)

    if (run_dir / "predictions.jsonl").exists():
        print(f"[{i}/{len(RUNS)}] {tag}: already present, skipping\n", flush=True)
        results.append((tag, "skipped", 0.0))
        continue

    cmd = [sys.executable, "-u", "/kaggle/working/EmoCap/scripts/train_arm.py",
           "--arm", arm, "--fold", str(fold), "--seed", str(SEED),
           "--data-root", DATA, "--out-root", OUT]
    if nc:
        cmd.append("--negative-control")

    print(f"[{i}/{len(RUNS)}] {tag}   ({(time.time()-t_all)/60:.1f} min into batch)",
          flush=True)
    t0 = time.time()
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print("   ", line, end="")
    rc = proc.wait()

    # A failure does NOT abort the batch. The runs are independent, and losing five good
    # ones because the fourth crashed is exactly the failure this loop exists to avoid.
    status = "ok" if rc == 0 else f"FAILED rc={rc}"
    results.append((tag, status, round((time.time() - t0) / 60, 1)))
    print(f"    -> {status}  [{results[-1][2]} min]\n", flush=True)


In [ ]:
# What landed. `empty_captions` above zero means decode collapsed and that run is suspect.
print(f"{'run':<28} {'status':<14} {'min':>6}")
for tag, status, mins in results:
    print(f"{tag:<28} {status:<14} {mins:>6.1f}")

for tag, status, _ in results:
    if status != "ok":
        continue
    st = json.loads(Path(OUT, tag, "stats.json").read_text())
    n = sum(1 for _ in Path(OUT, tag, "predictions.jsonl").open())
    print(f"\n{tag}: {n:,} decoded, {st['empty_captions']} empty, "
          f"loss {st['losses'][0]:.3f} -> {st['losses'][-1]:.3f}, "
          f"probe {st['novis_identical_rate']:.1%} unchanged, {st['wall_minutes']} min")
    # Distinctiveness is what diagnosed S_paired_matched, and the unique-caption count is
    # the cheap version of it available here. This arm has ONE register per image, so a low
    # count means the decoder collapsed rather than that the registers converged.
    print(f"    {st['unique_captions']:,} unique of {n:,} decoded")

failed = [t for t, s, _ in results if s.startswith("FAILED")]
if failed:
    raise RuntimeError(f"{len(failed)} run(s) failed: {failed}")
